In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))
from src.utils.config import load_config
from src.data.options_loader import load_options
from src.data.loaders import load_prices
from src.instruments.black_scholes import implied_vol_call
from src.instruments.european_option import EuropeanOption

In [12]:
df = load_options("AAPL", "2021-01-01", "2023-01-01")
prices = load_prices(["AAPL"], "2021-01-01", "2023-01-01").sort_index()
print(df[df['cp'] == 'C'].groupby("option_id").size().sort_values(ascending=False).head(10))

# Pick 1 Contract
id = df[df['cp'] == 'C'].groupby("option_id").size().sort_values(ascending=False).head(1).index[0]
contract = df[df["option_id"] == id].sort_values("date").set_index("date") # Important: set date as index

# Align prices and contract to the common dates
common_dates = contract.index.intersection(prices.index)
prices = prices.loc[common_dates]
contract = contract.loc[common_dates]
print("num obs in contract: ", len(contract))
print(contract[["c_date", "option_symbol", "option_id", "cp"]].head())
print(prices.head())

Cached prices available
Cached data covers request. Loading from cache.
option_id
115027241    40
115027239    40
115027237    40
118781573    40
107859905    39
114395851    39
114395847    39
114395849    39
110882415    39
110875573    39
dtype: int64
num obs in contract:  40
                c_date          option_symbol  option_id cp
2021-09-20  2021-09-20  AAPL  211119C00150000  115027241  C
2021-09-21  2021-09-21  AAPL  211119C00150000  115027241  C
2021-09-22  2021-09-22  AAPL  211119C00150000  115027241  C
2021-09-23  2021-09-23  AAPL  211119C00150000  115027241  C
2021-09-24  2021-09-24  AAPL  211119C00150000  115027241  C
Ticker            AAPL
2021-09-20  139.668350
2021-09-21  140.147141
2021-09-22  142.511765
2021-09-23  143.469315
2021-09-24  143.557266


In [13]:
port_cfg = load_config("../configs/portfolio.yaml")
tickers = port_cfg["portfolio"]["tickers"]
prices_full = load_prices(tickers, "2020-01-01", "2023-01-01").sort_index()
print(prices_full.head())

Cached prices available
Cached data covers request. Loading from cache.
Ticker           AAPL        MSFT       AMZN      GOOGL        META      NVDA  \
Date                                                                            
2020-01-02  72.400513  152.158401  94.900497  67.920815  208.324768  5.971077   
2020-01-03  71.696648  150.263733  93.748497  67.565498  207.222488  5.875504   
2020-01-06  72.267929  150.652130  95.143997  69.366386  211.125229  5.900145   
2020-01-07  71.928055  149.278534  95.343002  69.232399  211.582031  5.971576   
2020-01-08  73.085091  151.656311  94.598503  69.725174  213.727051  5.982776   

Ticker             JPM         UNH        XOM          PG  
Date                                                       
2020-01-02  119.036407  265.243103  53.306404  105.524033  
2020-01-03  117.465576  262.558838  52.877853  104.814331  
2020-01-06  117.372162  264.381592  53.283848  104.959656  
2020-01-07  115.376778  262.785522  52.847786  104.309799  


In [14]:
# Underlying price
S = prices["AAPL"]

# Expiry (constant per contract)
expiry = pd.to_datetime(contract["expiration_date"].iloc[0])

# Implied vol
iv = []

for t in contract.index:
    C = contract.loc[t, "price"]              # option price
    S_t = prices.loc[t, "AAPL"]               # spot
    K = contract["price_strike"].iloc[0]
    T_t = (expiry - t).days / 365.0

    if T_t <= 0:
        iv.append(np.nan)
        continue

    try:
        iv_t = implied_vol_call(C, S_t, K, r=0.03, T=T_t)
    except Exception as e:
        print(t, C, S_t, K, T_t, e)
        iv_t = np.nan

    iv.append(iv_t)

sigma = pd.Series(iv, index=contract.index)

# Time to maturity (vector)
T = pd.Series([ (expiry - t).days / 365.0 for t in contract.index ], index=contract.index)

# Build factor list
factors = []
for t in contract.index:
    factors.append({
        "spot": {"AAPL": S.loc[t]},
        "vol": {"AAPL": sigma.loc[t]},
        "rate": 0.03,
        "time": T.loc[t]
    })

In [15]:
factors

[{'spot': {'AAPL': np.float64(139.66835021972656)},
  'vol': {'AAPL': np.float64(0.32574779242924995)},
  'rate': 0.03,
  'time': np.float64(0.1643835616438356)},
 {'spot': {'AAPL': np.float64(140.1471405029297)},
  'vol': {'AAPL': np.float64(0.3123193775600172)},
  'rate': 0.03,
  'time': np.float64(0.16164383561643836)},
 {'spot': {'AAPL': np.float64(142.5117645263672)},
  'vol': {'AAPL': np.float64(0.309920218900335)},
  'rate': 0.03,
  'time': np.float64(0.1589041095890411)},
 {'spot': {'AAPL': np.float64(143.4693145751953)},
  'vol': {'AAPL': np.float64(0.29812310089462446)},
  'rate': 0.03,
  'time': np.float64(0.15616438356164383)},
 {'spot': {'AAPL': np.float64(143.55726623535156)},
  'vol': {'AAPL': np.float64(0.29590218426154097)},
  'rate': 0.03,
  'time': np.float64(0.15342465753424658)},
 {'spot': {'AAPL': np.float64(142.04273986816406)},
  'vol': {'AAPL': np.float64(0.2941865434589648)},
  'rate': 0.03,
  'time': np.float64(0.14520547945205478)},
 {'spot': {'AAPL': np.flo

In [16]:
option = EuropeanOption(
    ticker="AAPL",
    strike=contract["price_strike"].iloc[0],
    maturity=T.iloc[0],   # initial maturity (not used dynamically)
    option_type="call",   # since cp == 'C'
    quantity=1
)

Compute PnL

In [17]:
pnl = []

for i in range(1, len(factors)):
    f0 = factors[i-1]
    f1 = factors[i]
    pnl.append(option.pnl(f0, f1))

In [18]:
pnl

[np.float64(-0.1750002000000137),
 np.float64(0.7250001999998616),
 np.float64(0.05000040000014394),
 np.float64(-0.07500079999999798),
 np.float64(-0.7749995999999939),
 np.float64(-0.7199999999999918),
 np.float64(0.14000000000038426),
 np.float64(-0.4450000000004124),
 np.float64(0.030000000000015348),
 np.float64(-0.6999998999998489),
 np.float64(0.054999899999852886),
 np.float64(0.22499999999306652),
 np.float64(0.2050000000069403),
 np.float64(-0.27000000000132474),
 np.float64(-0.1349999999986693),
 np.float64(-0.2550000000000061),
 np.float64(-0.29500000000001947),
 np.float64(0.3300000000000125),
 np.float64(0.1299999999999173),
 np.float64(0.5200000000000671),
 np.float64(0.9850000000000065),
 np.float64(0.17500000000001847),
 np.float64(-0.02500000000001279),
 np.float64(-0.3250000000000739),
 np.float64(-0.1799998000004095),
 np.float64(0.20499980000047913),
 np.float64(0.049999999999968736),
 np.float64(1.6500004000000388),
 np.float64(-2.120000400000002),
 np.float64(-0.